In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np

C:\Users\HP\AppData\Local\Temp\ipykernel_18296\627819429.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [2]:
loader = PyPDFLoader('data/2312.10997v5.pdf')
loader = PyPDFLoader('data/10201979.pdf')
loader = PyPDFLoader('data/2509.23988v3.pdf')
loader = PyPDFLoader('data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf')
pages = loader.load()


### Text Cleaning

In [ ]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spaces
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    cleaned_text = cleaned_text.replace("__", "")
    return cleaned_text

In [4]:
for page in pages:
    page.page_content = cleanText(page.page_content)

In [5]:
pages[: 10]

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the Potential of Offline LLMs in Data Science: A Study on Code Generation for Data Analysis ANASTASIOS NIKOLAKOPOULOS 1, ANTONIOS LITKE 1, ALEXANDROS PSYCHAS 1, ELENI VERONI 2, AND THEODORA VARVARIGOU 1 1School of Electrical and Computer Engineering, Institute of

### Chunking
 - splitting text information into smaller sizes

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 200)

In [7]:
chunks = splitter.split_documents(pages)

In [8]:
chunks[0:10]

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the Potential of Offline LLMs in Data Science: A Study on Code Generation for Data Analysis ANASTASIOS NIKOLAKOPOULOS 1, ANTONIOS LITKE 1, ALEXANDROS PSYCHAS 1, ELENI VERONI 2, AND THEODORA VARVARIGOU 1 1School'),
 Document(metadata={'producer': 'pdfTeX-1.40.24',

In [9]:
len(chunks)

693

### Embeddings

In [10]:
def createEmbedding(text):
    response = ollama.embed(
    model = 'nomic-embed-text-v2-moe',
    input = text ,
    )
    return response.embeddings

In [29]:
sample_text = "this is a black car!"
print(createEmbedding(sample_text))

[[0.017525606, -0.055630695, -0.03781086, 0.0278525, 0.014276839, -0.006948967, -0.05646756, -0.018903894, -0.008746914, 0.0019043909, 0.028743535, -0.056393348, 0.07700931, 0.046605684, 0.0015968725, -0.036564734, 0.028226849, -0.0030354303, -0.008055718, 0.061202627, 0.045994826, -0.052638747, -0.03578447, -0.03279134, 0.04578182, -0.0076456754, 0.021769173, 0.021924606, 0.038195003, -0.0159788, 0.0032565072, -0.048310883, -0.0798269, 0.047762796, -0.07094089, 0.004436961, -0.005479182, -0.0046320767, 0.046030004, -0.022270374, 0.023837794, 0.012069714, 0.007284152, 0.06899126, 0.041998155, 0.015125231, -0.0146937845, 0.022237558, -0.019037833, 0.05459687, 0.02681172, 0.015743883, 0.02900423, -0.018440697, 0.041370295, -0.049441222, 0.012728367, -0.02120271, -0.0002063336, -0.0051484145, 0.025235225, 0.024403267, -0.006140415, -0.07623727, -0.06838044, 0.0023302925, 0.033804446, -0.009078112, 0.082351394, 0.009654663, -0.062019415, 0.03309109, -0.048259597, -0.0060315984, 0.008941451

In [30]:
doc_embeddings = []

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding)

In [31]:
len(doc_embeddings)

693

In [52]:
user_query = input('Ask anything about the Document...')


In [53]:
print(user_query)   

how does RAG enhances LLMs?


In [54]:
user_query_embedding  = createEmbedding(user_query)

In [56]:
print(user_query_embedding)

[[0.013303927, -0.074483775, -0.026553012, -0.0029712257, 0.04690742, -0.04753944, -0.0033687274, -0.0063986317, -0.020115923, 0.078084186, -0.08183789, -0.025722947, 0.0042209495, 0.01972109, -0.03656803, -0.06740721, -0.0077790716, 0.03743166, -0.042619463, 0.014035858, 0.0077002323, 0.030331708, -0.064098604, 0.013771542, -0.015219318, 0.058306452, -0.07731662, -0.009083607, 0.039683852, 0.010342302, 0.026210008, 0.018980296, 0.08769567, 0.042929817, 0.0125058815, 0.018191585, -0.04875399, -0.022715207, -0.020320171, -0.030413827, 0.039167717, 0.0014267545, 0.015911138, -0.05861516, -0.012581836, 0.0018110392, -0.12233331, -0.012262249, 0.00641009, 0.003411423, 0.009361511, 0.0067407275, 0.039396543, -0.07188285, 0.06235025, -0.049806524, -0.097283706, -0.063280396, 0.04475386, 0.0072719143, -0.037567195, -0.05624266, 0.018255066, 0.05037871, -0.036172286, 0.059801847, 0.097516984, 0.026020756, -0.03066414, 0.044089608, 0.008717549, 2.8502584e-05, 0.032753732, -0.011217738, -0.04075

In [57]:
above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embedding)
    if score >= 0.2:
        result = (index, score)
        above_zero_point_six.append(result)

In [58]:
above_zero_point_six

[(0, array([[0.31431209]])),
 (1, array([[0.29142793]])),
 (4, array([[0.36340738]])),
 (5, array([[0.38317341]])),
 (6, array([[0.37476884]])),
 (7, array([[0.32220121]])),
 (8, array([[0.31921123]])),
 (9, array([[0.33258326]])),
 (10, array([[0.3711628]])),
 (11, array([[0.3510522]])),
 (12, array([[0.41509976]])),
 (13, array([[0.44327589]])),
 (14, array([[0.43683224]])),
 (15, array([[0.45439738]])),
 (16, array([[0.45999845]])),
 (17, array([[0.45087625]])),
 (18, array([[0.41118982]])),
 (19, array([[0.41457373]])),
 (21, array([[0.39608567]])),
 (22, array([[0.37573887]])),
 (23, array([[0.38286504]])),
 (24, array([[0.41928772]])),
 (26, array([[0.32039084]])),
 (27, array([[0.27250226]])),
 (28, array([[0.35646489]])),
 (29, array([[0.41758518]])),
 (30, array([[0.3581481]])),
 (31, array([[0.39436999]])),
 (32, array([[0.35805067]])),
 (33, array([[0.3499211]])),
 (35, array([[0.33268379]])),
 (36, array([[0.29731069]])),
 (37, array([[0.27525971]])),
 (39, array([[0.331584

In [59]:
largest = max(above_zero_point_six, key=lambda x: x[1])
largest

(16, array([[0.45999845]]))

In [60]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

applications, with one of the main goals being the improved accuracy of the models’ outputs [3], [4]. Of course, the applications of LLMs go far beyond traditional natural language processing. In scientific research, for example, LLMs are used to speed up discoveries in medicine, biology, materials science, and forensics. For instance, their application in drug discovery could help enable speed


In [61]:
chunks[above_zero_point_six[2][0]].page_content

'the Horizon Europe Program through the Project ‘‘Reliable biomeTric tEchNologies to asSist Police authorities in cOmbating terrorism and oRganized crime (TENSOR)’’ under Grant 101073920. ABSTRACT Large Language Models (LLMs) have recently attracted considerable attention from the scientific community, due to their advanced capabilities and potential to serve as vital tools across various'

### Evaluations

In [62]:
a = np.array([1,2,3])
b = np.array([1,2,3])

In [63]:
dot_sim = np.dot(a,b)

In [64]:
cos_sim = np.dot(a,b) / (np.linalg.norm(a)* np.linalg.norm(b))

In [65]:
euclidean_dist = np.linalg.norm(a - b)

In [66]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

 
 DOT PRODUCT SIM_SCORE: 14  
 
 COSINE SIM_SCORE: 1.0  
 
 EUCLIDEAN DISTANCE: 0.0  
 



In [67]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [27]:
user_query = input("ask me anything...")